In [4]:
import ROOT as r
import math
import numpy as np
from itertools import combinations
from neutrinosolver import *

# opens file and tree
f = r.TFile("actual_data/ttbar_5k.root")
tree = f.Get("Events")

nEvents = tree.GetEntries()

# classes for muon, electron, jets

class MyMuon(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
    
    
    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyElectron(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0, cutBased=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
        self.cutBased = cutBased

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyJet(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, btag=0.0, jetid=False):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False

    def IsBTagged(self, threshold):
        return self.btag > threshold

    def HasJetID(self):
        return (self.jetid & 2) != 0
    
    

# histograms

h_Mwh_vs_Mth = r.TH2F(
    "h_Mwh_vs_Mth",
    "M(W_h) vs M(t_h); M(t_h) [GeV]; M(W_h) [GeV]",
    50, 0, 500,
    50, 0, 300
)

# Dn,min distributions
h_Dn_correct = r.TH1F(
    "h_Dn_correct",
    "Correct leptonic b;D_{n,min};Normalized events",
    50, 0, 150
)

h_Dn_wrong = r.TH1F(
    "h_Dn_wrong",
    "Wrong leptonic b;D_{n,min};Normalized events",
    50, 0, 150
)

# Hadronic b candidate distributions
h_Dn_bh_correct = r.TH1F(
    "h_Dn_bh_correct",
    "Correct hadronic b;D_{n,min};Normalized events",
    50, 0, 150
)

h_Dn_bh_wrong = r.TH1F(
    "h_Dn_bh_wrong",
    "Wrong hadronic b;D_{n,min};Normalized events",
    50, 0, 150
)

# bh + light jet mass
h_Mbhj_correct = r.TH1F(
    "h_Mbhj_correct",
    "Correct bh+jet;M(b_{h}+j) [GeV];Normalized events",
    50, 0, 500
)

h_Mbhj_wrong = r.TH1F(
    "h_Mbhj_wrong",
    "Wrong bh+jet;M(b_{h}+j) [GeV];Normalized events",
    50, 0, 500
)


# weighted error
h_Mwh_vs_Mth.Sumw2()

h_Dn_correct.Sumw2()
h_Dn_wrong.Sumw2()

h_Dn_bh_correct.Sumw2()
h_Dn_bh_wrong.Sumw2()

h_Mbhj_correct.Sumw2()
h_Mbhj_wrong.Sumw2()


# analysis cuts

cuts = {
    "Muon": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "iso_max": 0.15
    },
    "Electron": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "cutBased": 4,
        "iso_max": 0.15
        
    },
    "Jets": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "btag": 0.3040
    },
    "MET": {
        "pt_min": 20.0
    }
}

cuts["Trigger"] = {
    "muon": ["HLT_IsoMu27"],
    "electron": ["HLT_Ele27_WPTight_Gsf", "HLT_Ele32_WPTight_Gsf"]
}

cutflow = {
    "total": 0,

    # triggers
    "pass_mu_trigger": 0,
    "pass_ele_trigger": 0,

    # lepton selection
    "exactly_1_lepton": 0,

    # MET
    "pass_MET": 0,

    # jets
    "3jets": 0,
    "4plus_jets": 0,

    # b-tag categories
    "4pj_2b": 0,
    "4pj_1b": 0,
    "3j_2b": 0,
}

def make_gen_vector(idx):
    vec = r.TLorentzVector()
    vec.SetPtEtaPhiM(
        tree.GenPart_pt[idx],
        tree.GenPart_eta[idx],
        tree.GenPart_phi[idx],
        tree.GenPart_mass[idx]
    )
    return vec



def make_neutrino(nusol):
    nu_vec = nusol.nu

    nu = r.TLorentzVector()
    nu.SetPxPyPzE(
        nu_vec[0],
        nu_vec[1],
        nu_vec[2],
        math.sqrt(
            nu_vec[0]**2 +
            nu_vec[1]**2 +
            nu_vec[2]**2
        )
    )

    return nu

def get_last_copy(idx, mothers, pdg_ids):
        current = idx
        visited = set()

        while current not in visited:
            visited.add(current)

            same_children = [
                j for j, mother in enumerate(mothers)
                if mother == current
                and pdg_ids[j] == pdg_ids[current]
            ]

            if not same_children:
                return current

            current = same_children[0]

        return current

n_3jet_before_solver = 0
n_3jet_solver_success = 0
n_3jet_solver_fail = 0

# event loop
for event in range(nEvents):

    cutflow["total"] += 1

    tree.GetEntry(event)
    weight = tree.Generator_weight


    gen_b_had = None
    gen_b_lep = None
    gen_w_quarks = []
    gen_mothers = list(tree.GenPart_genPartIdxMother)
    gen_pdg_ids = list(tree.GenPart_pdgId)


    # Find the last-copy top and anti-top particles
    top_indices = []

    for i in range(tree.nGenPart):
        if abs(gen_pdg_ids[i]) == 6:
            top_indices.append(i)


    for top_idx in top_indices:

        # Direct top daughters
        top_daughters = [
            idx for idx, mother in enumerate(gen_mothers)
            if mother == top_idx
        ]

        b_candidates = [
            idx for idx in top_daughters
            if abs(gen_pdg_ids[idx]) == 5
        ]

        W_candidates = [
            idx for idx in top_daughters
            if abs(gen_pdg_ids[idx]) == 24
        ]

        if not b_candidates or not W_candidates:
            continue

        # Follow b and W through generator copies
        b_idx = get_last_copy(
            b_candidates[0],
            gen_mothers,
            gen_pdg_ids
        )

        W_idx = get_last_copy(
            W_candidates[0],
            gen_mothers,
            gen_pdg_ids
        )

        W_daughters = [
            idx for idx, mother in enumerate(gen_mothers)
            if mother == W_idx
        ]

        charged_leptons = [
            idx for idx in W_daughters
            if abs(gen_pdg_ids[idx]) in [11, 13, 15]
        ]

        neutrinos = [
            idx for idx in W_daughters
            if abs(gen_pdg_ids[idx]) in [12, 14, 16]
        ]

        quarks = [
            idx for idx in W_daughters
            if abs(gen_pdg_ids[idx]) in [1, 2, 3, 4]
        ]

        bvec = make_gen_vector(b_idx)

        # Leptonic top
        if charged_leptons and neutrinos:
            gen_b_lep = bvec

        # Hadronic top
        elif len(quarks) == 2:
            gen_b_had = bvec

            gen_w_quarks = [
                make_gen_vector(quarks[0]),
                make_gen_vector(quarks[1])
            ]
    
    if gen_b_had is None:
        continue

    if gen_b_lep is None:
        continue

    if len(gen_w_quarks) != 2:
        continue

    # MET filters
    if not all([
        tree.Flag_goodVertices,
        tree.Flag_globalSuperTightHalo2016Filter,
        tree.Flag_HBHENoiseFilter,
        tree.Flag_HBHENoiseIsoFilter,
        tree.Flag_EcalDeadCellTriggerPrimitiveFilter,
        tree.Flag_BadPFMuonFilter,
        tree.Flag_BadPFMuonDzFilter,
        tree.Flag_eeBadScFilter,
        tree.Flag_ecalBadCalibFilter
    ]):
        continue


    # trigger cuts
    passes_mu_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["muon"]
        )

    passes_ele_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["electron"]
        )   

    if passes_mu_trigger:
        cutflow["pass_mu_trigger"] += 1

    if passes_ele_trigger:
        cutflow["pass_ele_trigger"] += 1

    # muons

    muons = []
    for mu_idx in range(tree.nMuon):
        mu = MyMuon(
            tree.Muon_pt[mu_idx],
            tree.Muon_eta[mu_idx],
            tree.Muon_phi[mu_idx],
            tree.Muon_mass[mu_idx],   
            tree.Muon_miniPFRelIso_all[mu_idx],
            tree.Muon_charge[mu_idx] #unsure
            )
        muons.append(mu)

    #iso_muons = [m for m in muons if m.IsIsolated(MuonRelIsoCut)]

    iso_muons = [
    m for m in muons
    if m.Pt() > cuts["Muon"]["pt_min"]
    and abs(m.Eta()) < cuts["Muon"]["eta_max"]
    and m.isolation < cuts["Muon"]["iso_max"]
    ]



    iso_muons = sorted(iso_muons, key=lambda m: m.Pt(), reverse=True)

    #if len(iso_muons) >= 2 and iso_muons[0].Pt() > MuonPtCut:
        #dimu = iso_muons[0] + iso_muons[1]
        #h_Mmumu.Fill(dimu.M(), weight)


    # electrons
    electrons = []
    for ele_idx in range(tree.nElectron):
        ele = MyElectron(
            tree.Electron_pt[ele_idx],
            tree.Electron_eta[ele_idx],
            tree.Electron_phi[ele_idx],
            tree.Electron_mass[ele_idx],   
            tree.Electron_miniPFRelIso_all[ele_idx],  
            tree.Electron_charge[ele_idx],
            tree.Electron_cutBased[ele_idx]
            )
        electrons.append(ele)

    
    iso_electrons = [
        e for e in electrons
        if e.Pt() > cuts["Electron"]["pt_min"]
        and abs(e.Eta()) < cuts["Electron"]["eta_max"]
        and e.cutBased >= cuts["Electron"]["cutBased"]
        and e.isolation < cuts["Electron"]["iso_max"] 
        ]

    #iso_electrons = [e for e in electrons if e.IsIsolated(ElectronRelIsoCut)]
    iso_electrons = sorted(iso_electrons, key=lambda e: e.Pt(), reverse=True)

    #if len(iso_electrons) >= 2 and iso_electrons[0].Pt() > ElectronPtCut:
    #   diele = iso_electrons[0] + iso_electrons[1]
    #  h_Mee.Fill(diele.M(), weight)


    # MET from tree
    MET  = tree.MET_pt
    phi  = tree.MET_phi


    METx = MET * r.TMath.Cos(phi)
    METy = MET * r.TMath.Sin(phi)

    n_iso_mu  = len(iso_muons)
    n_iso_ele = len(iso_electrons)

    if n_iso_mu == 1 and n_iso_ele == 0:
        if not passes_mu_trigger:
            continue
        selected_lepton = iso_muons[0]

    elif n_iso_ele == 1 and n_iso_mu == 0:
        if not passes_ele_trigger:
            continue
        selected_lepton = iso_electrons[0]

    else:
        continue
    cutflow["exactly_1_lepton"] += 1
    passes_MET = MET > cuts["MET"]["pt_min"]

    if not passes_MET:
        continue
    cutflow["pass_MET"] += 1

    
    # jets

    jets = []
    for jet_idx in range(tree.nJet):
        jet = MyJet(
            tree.Jet_pt[jet_idx],
            tree.Jet_eta[jet_idx],
            tree.Jet_phi[jet_idx],
            tree.Jet_mass[jet_idx],    
            tree.Jet_btagDeepFlavB[jet_idx],    
            tree.Jet_jetId[jet_idx] 
            )
        jets.append(jet)

    good_jets = [
    j for j in jets
    if j.Pt() > cuts["Jets"]["pt_min"]
    and abs(j.Eta()) < cuts["Jets"]["eta_max"]
    and j.HasJetID()
    and j.DeltaR(selected_lepton) > 0.4
    ]

    #good_jets = [j for j in jets if j.HasJetID() and j.Pt() > JetPtCut]
    good_jets = sorted(good_jets, key=lambda j: j.Pt(), reverse=True)
    
    

    # flagging b-tagged jets
    for j in good_jets:
        j.is_btagged = j.IsBTagged(cuts["Jets"]["btag"])

    bjets = [j for j in good_jets if j.is_btagged]
    bjets = sorted(bjets, key=lambda j: j.Pt(), reverse=True)

    n_jets = len(good_jets)
    n_bjets = len(bjets)

    sigma2 = np.array([
        [100, 0],
        [0, 100]
    ])
    
    
    
    #4-6 jets
    if 4 <= n_jets <= 6:
        cutflow["4plus_jets"] += 1

        if n_bjets >= 2:
            cutflow["4pj_2b"] += 1

            b1, b2 = bjets[0], bjets[1]
            light_jets = [j for j in good_jets if not j.is_btagged]

            if len(light_jets) < 2:
                continue

            best_pair = None
            best_diff = float("inf")

            for j1_tmp, j2_tmp in combinations(light_jets, 2):
                W_tmp = j1_tmp + j2_tmp
                diff = abs(W_tmp.M() - 80.4)

                if diff < best_diff:
                    best_diff = diff
                    best_pair = (j1_tmp, j2_tmp)

            if best_pair is None:
                continue

            j1, j2 = best_pair
            W_had = j1 + j2

            for b_had in [b1, b2]:

                # Fill only the correctly matched hadronic b
                if b_had.DeltaR(gen_b_had) >= 0.4:
                    continue

                t_had = W_had + b_had

                h_Mwh_vs_Mth.Fill(
                    t_had.M(),
                    W_had.M(),
                    weight
                )

    # 3-jet reconstruction
    if n_jets == 3:
        cutflow["3jets"] += 1

        # exactly two b-tagged jets and one light jet
        if n_bjets != 2:
            continue

        cutflow["3j_2b"] += 1

        b1, b2 = bjets[0], bjets[1]
        light_jets = [j for j in good_jets if not j.is_btagged]

        if len(light_jets) != 1:
            continue

        n_3jet_before_solver += 1

        light_jet = light_jets[0]

        light_is_correct = (
            light_jet.DeltaR(gen_w_quarks[0]) < 0.4
            or light_jet.DeltaR(gen_w_quarks[1]) < 0.4
        )

        assignments = []

        for b_lep, b_had in [(b1, b2), (b2, b1)]:

            try:
                nusol = singleNeutrinoSolution(
                    b_lep,
                    selected_lepton,
                    METx,
                    METy,
                    sigma2
                )
            except Exception as error:
                n_3jet_solver_fail += 1
                continue

            n_3jet_solver_success += 1

            nu = make_neutrino(nusol)

            Dn = nusol.chi2
            Mbhj = (b_had + light_jet).M()
            if not np.isfinite(Dn):
                continue

            if not np.isfinite(Mbhj):
                continue

            b_lep_correct = b_lep.DeltaR(gen_b_lep) < 0.4
            b_had_correct = b_had.DeltaR(gen_b_had) < 0.4

            # Plot 1: Dn for correct/wrong leptonic b
            if b_lep_correct:
                h_Dn_correct.Fill(Dn)
            else:
                h_Dn_wrong.Fill(Dn)

            # Plot 2: Dn for correct/wrong hadronic b candidate
            if b_had_correct:
                h_Dn_bh_correct.Fill(Dn)
            else:
                h_Dn_bh_wrong.Fill(Dn)

            # Plot 3: mass of bh + light jet
            if b_had_correct and light_is_correct:
                h_Mbhj_correct.Fill(Mbhj)
            else:
                h_Mbhj_wrong.Fill(Mbhj)

            # Cuts used only for accepted reconstruction assignments
            if Dn > 150 or Mbhj > 500:
                continue

            assignments.append({
                "b_lep": b_lep,
                "b_had": b_had,
                "nu": nu,
                "Dn": Dn,
                "Mbhj": Mbhj
            })

        # No assignment produced a valid neutrino solution
        if len(assignments) == 0:
            continue

        # Only one b candidate produced a valid solution:
        # that candidate is the leptonic b.
        if len(assignments) == 1:
            best_assignment = assignments[0]

        # Both candidates produced solutions.
        # The likelihood selection must be done later using
        # probability templates from a separate first pass.
        else:
            best_assignment = None


def normalize_hist(hist):
    integral = hist.Integral()

    if integral > 0:
        hist.Scale(1.0 / integral)

print("Dn correct:", h_Dn_correct.GetEntries())
print("Dn wrong:", h_Dn_wrong.GetEntries())
print("Dn bh correct:", h_Dn_bh_correct.GetEntries())
print("Dn bh wrong:", h_Dn_bh_wrong.GetEntries())
print("Mass correct:", h_Mbhj_correct.GetEntries())
print("Mass wrong:", h_Mbhj_wrong.GetEntries())
print("3-jet events before solver:", n_3jet_before_solver)
print("3-jet solver successes:", n_3jet_solver_success)
print("3-jet solver failures:", n_3jet_solver_fail)

for hist in [
    h_Dn_correct,
    h_Dn_wrong,
    h_Dn_bh_correct,
    h_Dn_bh_wrong,
    h_Mbhj_correct,
    h_Mbhj_wrong
]:
    normalize_hist(hist)

# histograms

print("\n=== CUTFLOW ===")
for key, val in cutflow.items():
    print(f"{key:15s}: {val}")



#probability 2d hist

c_Mwh_vs_Mth = r.TCanvas("c_Mwh_vs_Mth", "M(W_h) vs M(t_h)", 800, 600)

if h_Mwh_vs_Mth.Integral() > 0:
    h_Mwh_vs_Mth.Scale(1.0 / h_Mwh_vs_Mth.Integral("width"))

def top_probability(M_top, M_W, hist):
    """
    Returns the probability density at (M_top, M_W)
    from the normalized 2D histogram.
    """

    xbin = hist.GetXaxis().FindBin(M_top)
    ybin = hist.GetYaxis().FindBin(M_W)

    return hist.GetBinContent(xbin, ybin)


#p = top_probability(173.2, 81.1, h_Mwh_vs_Mth)
#print(p)

h_Mwh_vs_Mth.Draw("COLZ")   # important for 2D histograms

c_Mwh_vs_Mth.Update()


import os
os.makedirs("plots", exist_ok=True)
c_Mwh_vs_Mth.SaveAs("plots/h_Mwh_vs_Mth.png")

canvases = []


def save_comparison(
    correct_hist,
    wrong_hist,
    canvas_name,
    output_name
):
    canvas = r.TCanvas(canvas_name, canvas_name, 800, 600)
    canvases.append(canvas)

    correct_hist.SetLineColor(r.kBlue)
    wrong_hist.SetLineColor(r.kRed)

    correct_hist.SetLineWidth(2)
    wrong_hist.SetLineWidth(2)

    if correct_hist.GetMaximum() >= wrong_hist.GetMaximum():
        correct_hist.Draw("HIST")
        wrong_hist.Draw("HIST SAME")
    else:
        wrong_hist.Draw("HIST")
        correct_hist.Draw("HIST SAME")

    legend = r.TLegend(0.65, 0.75, 0.88, 0.88)
    legend.AddEntry(correct_hist, "Correct", "l")
    legend.AddEntry(wrong_hist, "Wrong", "l")
    legend.Draw()

    canvas.Update()
    canvas.Draw()
    canvas.SaveAs(output_name)

    return canvas


c_Dn_leptonic = save_comparison(
    h_Dn_correct,
    h_Dn_wrong,
    "c_Dn_leptonic",
    "plots/Dn_leptonic_b.png"
)

c_Dn_hadronic = save_comparison(
    h_Dn_bh_correct,
    h_Dn_bh_wrong,
    "c_Dn_hadronic",
    "plots/Dn_hadronic_b.png"
)

c_Mbhj = save_comparison(
    h_Mbhj_correct,
    h_Mbhj_wrong,
    "c_Mbhj",
    "plots/Mbhj_3jet.png"
)


Dn correct: 130.0
Dn wrong: 99.0
Dn bh correct: 127.0
Dn bh wrong: 102.0
Mass correct: 105.0
Mass wrong: 124.0
3-jet events before solver: 140
3-jet solver successes: 229
3-jet solver failures: 51

=== CUTFLOW ===
total          : 5000
pass_mu_trigger: 1009
pass_ele_trigger: 812
exactly_1_lepton: 1523
pass_MET       : 1386
3jets          : 385
4plus_jets     : 741
4pj_2b         : 436
4pj_1b         : 0
3j_2b          : 140


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Mwh_vs_Mth
Info in <TCanvas::Print>: png file plots/h_Mwh_vs_Mth.png has been created
Info in <TCanvas::Print>: png file plots/Dn_leptonic_b.png has been created
Info in <TCanvas::Print>: png file plots/Dn_hadronic_b.png has been created
Info in <TCanvas::Print>: png file plots/Mbhj_3jet.png has been created
